# 07 - Insights Gold: ecommerce_clientes
**Squad 2 - Grupo 2**

## Objetivo
Gerar os insights de negócio de `ecommerce_clientes` a partir dos KPIs da Gold e da quarentena da Silver, e gravá-los como tabela para o dashboard do Looker Studio. A cada ciclo de polling os insights são recalculados e gravados no Delta (camada gold) e no SQL Server.

## Insights gerados
| Regra | Insight | Severidade |
|---|---|---|
| 6 | Novos clientes na janela de 10 minutos | `OK` (informativo) |
| 7 | Percentual de cadastros de provedores desconhecidos | `ALERTA` se maior que 30% |
| 7 | Provedor de e-mail com mais cadastros | `OK` (informativo) |
| 8 | Registros barrados por e-mail duplicado (quarentena) | `ALERTA` se maior que 0 |
| 9 | Tempo médio entre cadastro e primeira compra | `OK` (informativo) |
| 9 | Clientes com primeira compra anterior ao cadastro | `ALERTA` se maior que 0 |
| 10 | Registros com atualização anterior ao cadastro (quarentena) | `ALERTA` se maior que 0 |

Se a origem de um insight ainda não existir, ele aparece com severidade `SEM_DADOS`.

## Paleta de Cores (Padrão Visual para Looker Studio)
| Severidade | Cor | Hex |
|---|---|---|
| OK | Verde | `#2E7D32` |
| ALERTA | Amarelo | `#F2B705` |
| CRÍTICO | Vermelho | `#C0392B` |
| SEM_DADOS | Cinza | `#8C8C8C` |

## Estrutura das tabelas
`regra`, `insight`, `titulo`, `mensagem`, `valor`, `severidade`, `cor_status` e `calculado_em`.

O campo `cor_status` mapeia automaticamente a severidade para a paleta visual acima.

## Modo de gravação
- **`gold_insights_clientes`:** `overwrite`, sempre os insights do último ciclo.
- **`gold_insights_clientes_historico`:** `append`, os insights de cada ciclo, com `calculado_em`.

## Saídas
- **Delta (Gold):** `squad2/gold/ecommerce_clientes/<tabela>`
- **SQL Server:** schema `squad2`, mesmas tabelas (fonte do dashboard no Looker Studio)
- **Unity Catalog:** views em `workspace.default` (consumo direto via SQL Warehouse no Looker Studio)

## Views criadas para Looker Studio
| View | Descrição |
|---|---|
| `vw_insights_clientes` | Todos os insights de clientes com cor_status |
| `vw_insights_clientes_kpi` | Resumo KPI consolidado (cards) |
| `vw_provedor_lider_detalhe` | Provedores: quantidade e percentual (gráfico de barras) |
| `vw_clientes_compras` / `vw_clientes_compras_detalhado` | Cadastrados, sem compra e com compra |
| `vw_qualidade_clientes` | E-mails duplicados e distribuição do tempo até a 1ª compra |
| `vw_distribuicao_primeira_compra` | Faixas de tempo até a primeira compra |
| `vw_clientes_por_ano` | Cadastrados vs compraram por ano |
| `vw_compradores_por_ano` | Compradores por ano (visão dos pedidos) |
| `vw_fontes_compradores` | Auditoria de proveniência dos compradores |

Além das views, a célula de registro cria a tabela `workspace.default.gold_insights_clientes` no Unity Catalog (cópia acumulada dos insights).

## Automação
O job `insights_clientes_atualizacao_15min` roda este notebook a cada 15 minutos, mantendo a Gold (Delta e SQL Server) e as views sempre atualizadas para o Looker Studio.

> **Nota de schema:** o histórico no SQL Server tem 7 colunas (sem `cor_status`) — a cor é derivável de `severidade` no consumo. O histórico no Delta tem 8 colunas.

## Dependências
- Notebook 06: tabelas de KPI da Gold (regras 6, 7 e 9).
- Notebook 05: quarentena da Silver (regras 8 e 10).

Rode o 05 e o 06 antes, para os insights refletirem dados recentes.

## Como executar
Rodar as células em ordem. O notebook executa ciclos a cada 15 segundos, por até 90 segundos. Ao final, cria as views no Unity Catalog para o Looker Studio. Em produção, o job agendado cuida da atualização a cada 15 minutos.

## Setup do ambiente
Roda o notebook 00 para carregar as variáveis do `.env`.

In [0]:
%run ./00_setup_config

## Importação das bibliotecas
Funções e tipos do Spark e utilitários de tempo usados no polling.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import os
import time
from datetime import datetime, timedelta

## Credenciais e caminhos
Monta as opções OAuth do Spark, lê as credenciais do SQL Server do `.env` e define os caminhos de leitura (KPIs da Gold e quarentena da Silver) e de gravação (Gold). Nenhuma credencial é exibida.

In [0]:
# Credenciais OAuth do Spark, SQL Server e caminhos das camadas no ADLS

storage_account_name = "internshipdatalake"

adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": os.getenv("ADLS_CLIENT_ID"),
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": os.getenv("ADLS_CLIENT_SECRET"),
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": (
        f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token"
    )
}

sql_host = os.getenv("SQL_HOST")
sql_database = os.getenv("SQL_DATABASE")
sql_username = os.getenv("SQL_USERNAME")
sql_password = os.getenv("SQL_PASSWORD")
sql_schema = "squad2"

base_squad = f"abfss://squad2@{storage_account_name}.dfs.core.windows.net"

caminho_gold_base = f"{base_squad}/gold/ecommerce_clientes"                   # KPIs do notebook 06 e destino dos insights
caminho_quarentena = f"{base_squad}/grupo2/quarantine/ecommerce_clientes"     # quarentena do notebook 05

print(f"Gold        : {caminho_gold_base}/<tabela>")
print(f"Quarentena  : {caminho_quarentena}")

## Parâmetros dos insights
Concentra a janela dos alertas de quarentena, o limite de provedores desconhecidos, o esquema da tabela de insights e os parâmetros do polling.

In [0]:
# Parâmetros dos insights

JANELA_ALERTAS_MIN = 60                     # janela da quarentena para as regras 8 e 10
LIMITE_PROVEDOR_DESCONHECIDO = 0.30         # regra 7: alerta se passar de 30% dos cadastros da janela

MOTIVOS_EMAIL_DUPLICADO = ["EMAIL_DUPLICADO_NO_LOTE", "EMAIL_EXISTE_NA_SILVER"]      # regra 8
MOTIVO_ATUALIZACAO_ANTERIOR = ["DT_ATUALIZACAO_ANTERIOR_AO_CADASTRO"]                # regra 10

schema_insights = StructType([
    StructField("regra", IntegerType(), False),
    StructField("insight", StringType(), False),
    StructField("titulo", StringType(), False),
    StructField("mensagem", StringType(), False),
    StructField("valor", DoubleType(), True),
    StructField("severidade", StringType(), False),
    StructField("cor_status", StringType(), False),
])

# Polling (mesmo padrão dos notebooks 04 e 06)
INTERVALO_SEGUNDOS = 15
DURACAO_MAXIMA_SEGUNDOS = 90
MAX_FALHAS_CONSECUTIVAS = 5

## Funções de apoio
`tabela_delta_existe` e `ler_delta` leem uma Delta por caminho (devolvendo `None` se ela ainda não existir). `materializar` calcula o resultado uma única vez, para que o Delta e o SQL Server recebam exatamente os mesmos valores. `gravar_gold` grava no Delta e no SQL Server com o mesmo modo.

In [0]:
def tabela_delta_existe(caminho):
    """Retorna True se já existe uma tabela Delta no caminho."""
    try:
        spark.read.format("delta").options(**adls_options).load(caminho).limit(1).count()
        return True
    except Exception:
        return False


def ler_delta(caminho):
    """Lê uma Delta por caminho. Devolve None se ela ainda não existir."""
    if not tabela_delta_existe(caminho):
        return None
    return spark.read.format("delta").options(**adls_options).load(caminho)


def materializar(df):
    """Calcula o resultado uma vez (poucas linhas) e devolve o DataFrame fixo e as linhas."""
    linhas = df.collect()
    return spark.createDataFrame(linhas, df.schema), linhas


def gravar_gold(df, nome_tabela, modo):
    """Grava no Delta (camada gold) e no SQL Server com o mesmo modo (overwrite ou append)."""
    escritor = df.write.format("delta").options(**adls_options).mode(modo)
    if modo == "overwrite":
        escritor = escritor.option("overwriteSchema", "true")
    escritor.save(f"{caminho_gold_base}/{nome_tabela}")

    (df.write
        .format("sqlserver")
        .option("host", sql_host)
        .option("port", 1433)
        .option("database", sql_database)
        .option("dbtable", f"{sql_schema}.{nome_tabela}")
        .option("user", sql_username)
        .option("password", sql_password)
        .option("encrypt", "true")
        .option("trustServerCertificate", "true")
        .mode(modo)
        .save())

## Construção dos insights
Cada regra gera um ou dois insights, no formato `(regra, insight, titulo, mensagem, valor, severidade)`.
- **Regras 6, 7 e 9:** leem as tabelas de KPI que o notebook 06 gravou na Gold.
- **Regras 8 e 10:** contam, na quarentena da Silver, os registros rejeitados dentro da janela de alertas.

Se a origem ainda não existir, o insight sai com severidade `SEM_DADOS`.

In [0]:
def novo_insight(regra, insight, titulo, mensagem, valor, severidade):
    cor = {"OK": "#2E7D32", "ALERTA": "#F2B705", "CRÍTICO": "#C0392B", "SEM_DADOS": "#8C8C8C"}.get(severidade, "#8C8C8C")
    return (regra, insight, titulo, mensagem, None if valor is None else float(valor), severidade, cor)


def sem_dados(regra, insight, titulo, origem):
    return novo_insight(regra, insight, titulo, f"{origem} ainda não disponível.", None, "SEM_DADOS")


def contar_quarentena(motivos):
    """Conta os registros da quarentena com o motivo dado dentro da janela de alertas."""
    df = ler_delta(caminho_quarentena)
    if df is None:
        return 0
    limite = F.expr(f"current_timestamp() - INTERVAL {JANELA_ALERTAS_MIN} MINUTES")
    return df.filter(F.col("motivo_rejeicao").isin(motivos) & (F.col("silver_processed_at") >= limite)).count()


def construir_insights():
    insights = []

    # Regra 6: novos clientes na janela
    df = ler_delta(f"{caminho_gold_base}/gold_kpi_novos_clientes_10min")
    linhas = df.collect() if df is not None else []
    if linhas:
        n, janela = linhas[0]["novos_clientes"], linhas[0]["janela_minutos"]
        insights.append(novo_insight(6, "novos_clientes", "Novos clientes",
            f"{n} novo(s) cliente(s) cadastrado(s) nos últimos {janela} minutos", n, "OK"))
    else:
        insights.append(sem_dados(6, "novos_clientes", "Novos clientes", "KPI 6 (notebook 06)"))

    # Regra 7: provedores de e-mail
    df = ler_delta(f"{caminho_gold_base}/gold_kpi_cadastros_por_provedor")
    linhas = df.collect() if df is not None else None
    if linhas is None:
        insights.append(sem_dados(7, "provedores_desconhecidos", "Provedores desconhecidos", "KPI 7 (notebook 06)"))
        insights.append(sem_dados(7, "provedor_lider", "Provedor com mais cadastros", "KPI 7 (notebook 06)"))
    elif not linhas:
        insights.append(novo_insight(7, "provedores_desconhecidos", "Provedores desconhecidos",
            "Nenhum cadastro na janela do KPI 7", 0, "OK"))
        insights.append(novo_insight(7, "provedor_lider", "Provedor com mais cadastros",
            "Nenhum cadastro na janela do KPI 7", None, "OK"))
    else:
        total = sum(r["cadastros"] for r in linhas)
        desconhecidos = sum(r["cadastros"] for r in linhas if not r["provedor_conhecido"])
        pct = desconhecidos * 100.0 / total
        janela = linhas[0]["janela_minutos"]
        severidade = "ALERTA" if pct > LIMITE_PROVEDOR_DESCONHECIDO * 100 else "OK"
        insights.append(novo_insight(7, "provedores_desconhecidos", "Provedores desconhecidos",
            f"{pct:.1f}% dos {total} cadastros dos últimos {janela} minutos vieram de provedores desconhecidos", pct, severidade))
        lider = max(linhas, key=lambda r: r["cadastros"])
        insights.append(novo_insight(7, "provedor_lider", "Provedor com mais cadastros",
            f"{lider['provedor']} lidera com {lider['cadastros']} cadastro(s) ({lider['taxa_pct']}%)", lider["taxa_pct"], "OK"))

    # Regra 8: e-mail duplicado (quarentena)
    qtd = contar_quarentena(MOTIVOS_EMAIL_DUPLICADO)
    insights.append(novo_insight(8, "email_duplicado", "E-mails duplicados",
        f"{qtd} registro(s) barrado(s) por e-mail duplicado nos últimos {JANELA_ALERTAS_MIN} minutos",
        qtd, "ALERTA" if qtd > 0 else "OK"))

    # Regra 9: tempo até a primeira compra
    df = ler_delta(f"{caminho_gold_base}/gold_kpi_tempo_primeira_compra")
    linhas = df.collect() if df is not None else []
    if linhas:
        t = linhas[0]
        if t["tempo_medio_dias"] is not None:
            insights.append(novo_insight(9, "tempo_primeira_compra", "Tempo até a primeira compra",
                f"Em média {t['tempo_medio_dias']} dia(s) entre o cadastro e a primeira compra ({t['clientes_com_compra']} clientes com compra)",
                t["tempo_medio_dias"], "OK"))
        else:
            insights.append(sem_dados(9, "tempo_primeira_compra", "Tempo até a primeira compra", "Clientes com compra"))
        antes = t["compra_antes_cadastro"]
        insights.append(novo_insight(9, "compra_antes_cadastro", "Compra antes do cadastro",
            f"{antes} cliente(s) com primeira compra anterior ao cadastro", antes, "ALERTA" if antes > 0 else "OK"))
    else:
        insights.append(sem_dados(9, "tempo_primeira_compra", "Tempo até a primeira compra", "KPI 9 (notebook 06)"))
        insights.append(sem_dados(9, "compra_antes_cadastro", "Compra antes do cadastro", "KPI 9 (notebook 06)"))

    # Regra 10: atualização anterior ao cadastro (quarentena)
    qtd = contar_quarentena(MOTIVO_ATUALIZACAO_ANTERIOR)
    insights.append(novo_insight(10, "atualizacao_anterior_cadastro", "Atualização anterior ao cadastro",
        f"{qtd} registro(s) com atualização anterior ao cadastro nos últimos {JANELA_ALERTAS_MIN} minutos",
        qtd, "ALERTA" if qtd > 0 else "OK"))

    return insights

## Ciclo de cálculo e gravação
Constrói os insights, calcula `calculado_em` e grava no Delta e no SQL Server: a tabela de insights é sobrescrita e o histórico recebe os insights do ciclo. No final imprime cada insight com a sua severidade.

In [0]:
def executar_ciclo():
    insights = construir_insights()

    df = (spark.createDataFrame(insights, schema_insights)
        .withColumns({"calculado_em": F.current_timestamp()}))
    df_fixo, linhas = materializar(df)

    gravar_gold(df_fixo, "gold_insights_clientes", "overwrite")            # sempre os insights do último ciclo

    # Histórico no Delta: schema completo com cor_status (tabela migrada, 8 colunas)
    (df_fixo.write
        .format("delta")
        .options(**adls_options)
        .mode("append")
        .save(f"{caminho_gold_base}/gold_insights_clientes_historico"))

    # Histórico no SQL Server: sem cor_status — a tabela foi criada com 7 colunas e a
    # Federation não permite DDL em foreign tables. A cor é derivável de severidade no consumo.
    (df_fixo.select("regra", "insight", "titulo", "mensagem", "valor", "severidade", "calculado_em")
        .write
        .format("sqlserver")
        .option("host", sql_host)
        .option("port", 1433)
        .option("database", sql_database)
        .option("dbtable", f"{sql_schema}.gold_insights_clientes_historico")
        .option("user", sql_username)
        .option("password", sql_password)
        .option("encrypt", "true")
        .option("trustServerCertificate", "true")
        .mode("append")
        .save())
    print(f"    [OK] gold_insights_clientes: {len(linhas)} insight(s) gravado(s) (Delta e SQL Server)")

    for r in linhas:
        print(f"    [{r['severidade']}] Regra {r['regra']} - {r['mensagem']}")

## Polling
Executa o ciclo em loop, a cada 15 segundos, por até 90 segundos. Encerra antes se houver 5 falhas consecutivas ou interrupção manual.

In [0]:
inicio = datetime.now()
fim_previsto = inicio + timedelta(seconds=DURACAO_MAXIMA_SEGUNDOS)
falhas_consecutivas = 0

print(f"Iniciando polling às {inicio.strftime('%H:%M:%S')}.")
print(f"Vai rodar até {fim_previsto.strftime('%H:%M:%S')} (ou até interromper manualmente).\n")

while datetime.now() < fim_previsto:
    try:
        print(f"--- Ciclo às {datetime.now().strftime('%H:%M:%S')} ---")
        executar_ciclo()
        falhas_consecutivas = 0
    except KeyboardInterrupt:
        print("\nPolling interrompido manualmente.")
        break
    except Exception as e:
        falhas_consecutivas += 1
        print(f"    [ERRO] Falha no ciclo ({falhas_consecutivas}/{MAX_FALHAS_CONSECUTIVAS}): {e}")
        if falhas_consecutivas >= MAX_FALHAS_CONSECUTIVAS:
            print(f"    {MAX_FALHAS_CONSECUTIVAS} falhas seguidas, interrompendo o loop.")
            break
    time.sleep(INTERVALO_SEGUNDOS)

print(f"\nPolling finalizado às {datetime.now().strftime('%H:%M:%S')}.")

## Conferência dos insights
Mostra a tabela de insights lida do Delta, a quantidade de linhas do histórico e uma leitura de volta do SQL Server, que é a tabela usada pelo Looker Studio.

In [0]:
print("=== gold_insights_clientes (Delta) ===")
df_insights = ler_delta(f"{caminho_gold_base}/gold_insights_clientes")
if df_insights is not None:
    display(df_insights.orderBy("regra", "insight"))
else:
    print("ainda não existe")

df_hist = ler_delta(f"{caminho_gold_base}/gold_insights_clientes_historico")
if df_hist is not None:
    print(f"Histórico: {df_hist.count()} linha(s)\n")

print("=== gold_insights_clientes (SQL Server) ===")
display(spark.read
    .format("sqlserver")
    .option("host", sql_host)
    .option("port", 1433)
    .option("database", sql_database)
    .option("dbtable", f"{sql_schema}.gold_insights_clientes")
    .option("user", sql_username)
    .option("password", sql_password)
    .option("encrypt", "true")
    .option("trustServerCertificate", "true")
    .load())

## Views para Looker Studio
Registra os insights no Unity Catalog (`workspace.default`) para consumo direto via SQL Warehouse no Looker Studio.

**Paleta de cores aplicada:**
| Severidade | Cor | Hex |
|---|---|---|
| OK | Verde | `#2E7D32` |
| ALERTA | Amarelo | `#F2B705` |
| CRÍTICO | Vermelho | `#C0392B` |
| SEM_DADOS | Cinza | `#8C8C8C` |

In [0]:
# ==============================================================================
# Looker Studio: registrar insights no Unity Catalog (workspace.default)
# Deriva os insights diretamente das tabelas UC (silver + quarentena + pedidos),
# sem depender do ADLS. Cria a tabela gold_insights_clientes no Unity Catalog.
# ==============================================================================

spark.sql("""
CREATE OR REPLACE TABLE workspace.default.gold_insights_clientes AS

-- Regra 6: Novos clientes cadastrados
SELECT 
    6 AS regra,
    'novos_clientes' AS insight,
    'Novos clientes' AS titulo,
    CONCAT(CAST(COUNT(*) AS STRING), ' cliente(s) cadastrado(s)') AS mensagem,
    CAST(COUNT(*) AS DOUBLE) AS valor,
    'OK' AS severidade,
    '#2E7D32' AS cor_status,
    current_timestamp() AS calculado_em
FROM workspace.default.silver_ecommerce_clientes

UNION ALL

-- Regra 7a: Percentual de provedores desconhecidos
SELECT 
    7 AS regra,
    'provedores_desconhecidos' AS insight,
    'Provedores desconhecidos' AS titulo,
    CONCAT(CAST(COALESCE(ROUND(t.pct_unknown, 1), 0) AS STRING),
           '% dos cadastros vieram de provedores desconhecidos') AS mensagem,
    CAST(COALESCE(t.pct_unknown, 0) AS DOUBLE) AS valor,
    CASE WHEN COALESCE(t.pct_unknown, 0) > 30 THEN 'ALERTA' ELSE 'OK' END AS severidade,
    CASE WHEN COALESCE(t.pct_unknown, 0) > 30 THEN '#F2B705' ELSE '#2E7D32' END AS cor_status,
    current_timestamp() AS calculado_em
FROM (
    SELECT 
        COUNT(CASE WHEN LOWER(SPLIT(email, '@')[1]) NOT IN ('gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com') THEN 1 END) * 100.0 / NULLIF(COUNT(*), 0) AS pct_unknown
    FROM workspace.default.silver_ecommerce_clientes
) t

UNION ALL

-- Regra 7b: Provedor com mais cadastros
SELECT 
    7 AS regra,
    'provedor_lider' AS insight,
    'Provedor com mais cadastros' AS titulo,
    CONCAT(COALESCE(t.provedor, 'Nenhum'), ' lidera com ',
           CAST(COALESCE(t.qtd, 0) AS STRING), ' cadastro(s) (',
           CAST(COALESCE(ROUND(t.qtd * 100.0 / total.total_qtd, 1), 0) AS STRING), '%)') AS mensagem,
    CAST(COALESCE(t.qtd, 0) AS DOUBLE) AS valor,
    'OK' AS severidade,
    '#2E7D32' AS cor_status,
    current_timestamp() AS calculado_em
FROM (
    SELECT 
        LOWER(SPLIT(email, '@')[1]) AS provedor,
        COUNT(*) AS qtd,
        ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS rn
    FROM workspace.default.silver_ecommerce_clientes
    GROUP BY LOWER(SPLIT(email, '@')[1])
) t
CROSS JOIN (SELECT COUNT(*) AS total_qtd FROM workspace.default.silver_ecommerce_clientes) total
WHERE t.rn = 1

UNION ALL

-- Regra 8: E-mails duplicados (quarentena)
SELECT 
    8 AS regra,
    'email_duplicado' AS insight,
    'E-mails duplicados' AS titulo,
    CONCAT(CAST(COUNT(*) AS STRING), ' registro(s) barrado(s) por e-mail duplicado') AS mensagem,
    CAST(COUNT(*) AS DOUBLE) AS valor,
    CASE WHEN COUNT(*) > 0 THEN 'ALERTA' ELSE 'OK' END AS severidade,
    CASE WHEN COUNT(*) > 0 THEN '#F2B705' ELSE '#2E7D32' END AS cor_status,
    current_timestamp() AS calculado_em
FROM workspace.default.quarentena_ecommerce_clientes
WHERE motivo_rejeicao IN ('EMAIL_DUPLICADO_NO_LOTE', 'EMAIL_EXISTE_NA_SILVER')

UNION ALL

-- Regra 9a: Tempo médio até a primeira compra
SELECT 
    9 AS regra,
    'tempo_primeira_compra' AS insight,
    'Tempo até a primeira compra' AS titulo,
    CONCAT('Em média ', CAST(COALESCE(ROUND(t.tempo_medio, 1), 0) AS STRING),
           ' dia(s) entre cadastro e primeira compra (',
           CAST(COALESCE(t.clientes_com_compra, 0) AS STRING), ' clientes)') AS mensagem,
    CAST(COALESCE(t.tempo_medio, 0) AS DOUBLE) AS valor,
    'OK' AS severidade,
    '#2E7D32' AS cor_status,
    current_timestamp() AS calculado_em
FROM (
    SELECT 
        AVG(dias_ate_compra) AS tempo_medio,
        COUNT(DISTINCT id_cliente) AS clientes_com_compra
    FROM (
        SELECT 
            c.id_cliente,
            DATEDIFF(MIN(p.dt_pedido), c.dt_cadastro) AS dias_ate_compra
        FROM workspace.default.silver_ecommerce_clientes c
        JOIN workspace.default.silver_ecommerce_pedidos p ON c.id_cliente = p.id_cliente
        GROUP BY c.id_cliente, c.dt_cadastro
    ) first_purchase
) t

UNION ALL

-- Regra 9b: Clientes com compra anterior ao cadastro
SELECT 
    9 AS regra,
    'compra_antes_cadastro' AS insight,
    'Compra antes do cadastro' AS titulo,
    CONCAT(CAST(COUNT(*) AS STRING), ' cliente(s) com primeira compra anterior ao cadastro') AS mensagem,
    CAST(COUNT(*) AS DOUBLE) AS valor,
    CASE WHEN COUNT(*) > 0 THEN 'ALERTA' ELSE 'OK' END AS severidade,
    CASE WHEN COUNT(*) > 0 THEN '#F2B705' ELSE '#2E7D32' END AS cor_status,
    current_timestamp() AS calculado_em
FROM (
    SELECT c.id_cliente, MIN(p.dt_pedido) AS primeira_compra, c.dt_cadastro
    FROM workspace.default.silver_ecommerce_clientes c
    JOIN workspace.default.silver_ecommerce_pedidos p ON c.id_cliente = p.id_cliente
    GROUP BY c.id_cliente, c.dt_cadastro
    HAVING MIN(p.dt_pedido) < c.dt_cadastro
) sub

UNION ALL

-- Regra 10: Atualização anterior ao cadastro (quarentena)
SELECT 
    10 AS regra,
    'atualizacao_anterior_cadastro' AS insight,
    'Atualização anterior ao cadastro' AS titulo,
    CONCAT(CAST(COUNT(*) AS STRING), ' registro(s) com atualização anterior ao cadastro') AS mensagem,
    CAST(COUNT(*) AS DOUBLE) AS valor,
    CASE WHEN COUNT(*) > 0 THEN 'ALERTA' ELSE 'OK' END AS severidade,
    CASE WHEN COUNT(*) > 0 THEN '#F2B705' ELSE '#2E7D32' END AS cor_status,
    current_timestamp() AS calculado_em
FROM workspace.default.quarentena_ecommerce_clientes
WHERE motivo_rejeicao = 'DT_ATUALIZACAO_ANTERIOR_AO_CADASTRO'
""")

print("[OK] Tabela workspace.default.gold_insights_clientes criada no Unity Catalog")
display(spark.table("workspace.default.gold_insights_clientes").orderBy("regra", "insight"))

In [0]:
%sql
-- ============================================================================
-- Views para Looker Studio — Insights de Clientes
-- Paleta: Verde #2E7D32 (OK), Amarelo #F2B705 (ALERTA), 
--         Vermelho #C0392B (CRÍTICO), Cinza #8C8C8C (SEM_DADOS)
-- ============================================================================

-- View 1: Todos os insights de clientes com cor_status
CREATE OR REPLACE VIEW workspace.default.vw_insights_clientes AS
SELECT
    regra,
    insight,
    titulo,
    mensagem,
    valor,
    severidade,
    cor_status,
    calculado_em
FROM workspace.default.gold_insights_clientes;

-- View 2: Resumo KPI consolidado para cards no Looker Studio
CREATE OR REPLACE VIEW workspace.default.vw_insights_clientes_kpi AS
SELECT
    titulo AS kpi,
    CAST(valor AS DECIMAL(18,2)) AS valor_destaque,
    mensagem AS descricao,
    severidade AS classificacao,
    cor_status AS cor
FROM workspace.default.gold_insights_clientes;

-- View 3: Detalhe do provedor líder (quantidade + percentual para gráfico)
CREATE OR REPLACE VIEW workspace.default.vw_provedor_lider_detalhe AS
SELECT
    LOWER(SPLIT(email, '@')[1]) AS provedor,
    COUNT(*) AS quantidade,
    ROUND(COUNT(*) * 100.0 / NULLIF((SELECT COUNT(*) FROM workspace.default.silver_ecommerce_clientes), 0), 1) AS percentual
FROM workspace.default.silver_ecommerce_clientes
GROUP BY LOWER(SPLIT(email, '@')[1])
ORDER BY quantidade DESC;

-- Prévia
SELECT * FROM workspace.default.vw_provedor_lider_detalhe LIMIT 10;

In [0]:
%sql
SHOW TABLES IN workspace.default;

In [0]:
%sql
SELECT * FROM workspace.default.vw_insights_clientes LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.vw_clientes_compras AS
SELECT
  COUNT(*) AS total_cadastrados,
  COUNT(*) - COUNT(DISTINCT CASE WHEN p.id_cliente IS NOT NULL THEN c.id_cliente END) AS sem_compra,
  COUNT(DISTINCT CASE WHEN p.id_cliente IS NOT NULL THEN c.id_cliente END) AS comprou
FROM workspace.default.silver_ecommerce_clientes c
LEFT JOIN workspace.default.silver_ecommerce_pedidos p
  ON c.id_cliente = p.id_cliente;

In [0]:
%sql
SELECT * FROM workspace.default.vw_clientes_compras

In [0]:
%sql
-- Proporção de e-mails duplicados e distribuição de tempo até a primeira compra
CREATE OR REPLACE VIEW workspace.default.vw_qualidade_clientes AS
SELECT
  (SELECT COUNT(*) FROM workspace.default.quarentena_ecommerce_clientes WHERE motivo_rejeicao IN ('EMAIL_DUPLICADO_NO_LOTE', 'EMAIL_EXISTE_NA_SILVER')) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM workspace.default.silver_ecommerce_clientes), 0) AS pct_email_duplicado,
  (SELECT AVG(datediff(min_pedido, dt_cadastro)) FROM (
     SELECT c.dt_cadastro, MIN(p.dt_pedido) AS min_pedido
     FROM workspace.default.silver_ecommerce_clientes c
     JOIN workspace.default.silver_ecommerce_pedidos p ON c.id_cliente = p.id_cliente
     GROUP BY c.id_cliente, c.dt_cadastro
  )) AS media_dias_primeira_compra;

-- Distribuição (faixa) de tempo até a primeira compra
CREATE OR REPLACE VIEW workspace.default.vw_distribuicao_primeira_compra AS
SELECT
  CASE 
    WHEN dias_ate_compra < 7 THEN 'Menos de 7 dias'
    WHEN dias_ate_compra < 30 THEN '7-29 dias'
    WHEN dias_ate_compra < 90 THEN '30-89 dias'
    WHEN dias_ate_compra < 365 THEN '90-364 dias'
    ELSE '365+ dias'
  END AS faixa_tempo,
  COUNT(*) AS qtd
FROM (
  SELECT DATEDIFF(MIN(p.dt_pedido), c.dt_cadastro) AS dias_ate_compra
  FROM workspace.default.silver_ecommerce_clientes c
  JOIN workspace.default.silver_ecommerce_pedidos p ON c.id_cliente = p.id_cliente
  GROUP BY c.id_cliente, c.dt_cadastro
) t
GROUP BY faixa_tempo
ORDER BY qtd DESC;

-- Prévia das views
SELECT * FROM workspace.default.vw_qualidade_clientes;
SELECT * FROM workspace.default.vw_distribuicao_primeira_compra;

## Clientes por ano: cadastrados vs compraram
Visão histórica desde o início: em cada ano, quantos clientes se cadastraram (`dt_cadastro`) e quantos desses têm pelo menos um pedido na Silver (`dt_pedido`).

A view `vw_clientes_por_ano` segue o mesmo critério da `vw_clientes_compras` (join por `id_cliente`), pronta para gráfico de colunas duplas no Looker Studio (dimensão: `ano`; métricas: `cadastrados` e `compraram`).

Como apenas **4 compradores** batem com um cadastro (ver aviso na próxima seção), a coluna `compraram` mostra valores baixos (0 a 2 por ano) — é o comportamento esperado, não um erro da query.

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.vw_clientes_por_ano AS
SELECT
  YEAR(c.dt_cadastro) AS ano,
  COUNT(*) AS cadastrados,
  COUNT(DISTINCT CASE WHEN p.id_cliente IS NOT NULL THEN c.id_cliente END) AS compraram
FROM workspace.default.silver_ecommerce_clientes c
LEFT JOIN workspace.default.silver_ecommerce_pedidos p
  ON c.id_cliente = p.id_cliente
GROUP BY YEAR(c.dt_cadastro)
ORDER BY ano;

SELECT * FROM workspace.default.vw_clientes_por_ano;

## Compradores por ano (visão dos pedidos)
Complemento da visão anterior: como os `id_cliente` dos pedidos não batem com os cadastros (base de pedidos independente, 10.002 clientes distintos), esta visão conta os compradores pelo ano da **primeira compra** direto na Silver de pedidos — sem join com cadastros.

⚠️ **AVISO PARA A EQUIPE:** dos 10.002 compradores dos pedidos, **9.998 NÃO têm cadastro** em nenhuma tabela de clientes do projeto — o `id_cliente` dos pedidos foi gerado em escala diferente dos cadastros (stream sintética independente). Apenas **4 compradores** correspondem a um cliente cadastrado. No Looker Studio, rotular a série como **"Compradores (pedidos)"** para não confundir com "clientes cadastrados". A query de proveniência no fim do notebook documenta as tabelas testadas.

A view `vw_compradores_por_ano` serve para comparar as duas séries no Looker Studio (dimensão: `ano`; métricas: `compraram`). Base maior, mas sem ligação com o cadastro.

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.vw_compradores_por_ano AS
SELECT
  YEAR(primeira_compra) AS ano,
  COUNT(*) AS compraram
FROM (
  SELECT id_cliente, MIN(dt_pedido) AS primeira_compra
  FROM workspace.default.silver_ecommerce_pedidos
  GROUP BY id_cliente
)
GROUP BY YEAR(primeira_compra)
ORDER BY ano;

SELECT * FROM workspace.default.vw_compradores_por_ano;

## Proveniência: fontes testadas para os compradores
Auditoria do cruzamento: cada tabela de clientes disponível no projeto foi testada contra os 10.002 compradores distintos da Silver de pedidos. O resultado confirma que apenas 4 compradores têm cadastro — a view `vw_fontes_compradores` registra essa auditoria (pode ser exibida no Looker Studio).

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.vw_fontes_compradores AS
SELECT * FROM (
  SELECT
    'workspace.default.silver_ecommerce_clientes' AS tabela_fonte,
    COUNT(*) AS total_clientes,
    MIN(c.id_cliente) AS id_min,
    MAX(c.id_cliente) AS id_max,
    COUNT(DISTINCT p.id_cliente) AS compradores_com_cadastro
  FROM workspace.default.silver_ecommerce_clientes c
  LEFT JOIN (SELECT DISTINCT id_cliente FROM workspace.default.silver_ecommerce_pedidos) p
    ON c.id_cliente = p.id_cliente

  UNION ALL

  SELECT
    'sqlserver_catalog.squad2.ecommerce_clientes' AS tabela_fonte,
    COUNT(*) AS total_clientes,
    MIN(c.id_cliente) AS id_min,
    MAX(c.id_cliente) AS id_max,
    COUNT(DISTINCT p.id_cliente) AS compradores_com_cadastro
  FROM sqlserver_catalog.squad2.ecommerce_clientes c
  LEFT JOIN (SELECT DISTINCT id_cliente FROM workspace.default.silver_ecommerce_pedidos) p
    ON c.id_cliente = p.id_cliente

  UNION ALL

  SELECT
    'workspace.default.ecommerce_clientes' AS tabela_fonte,
    COUNT(*) AS total_clientes,
    MIN(c.id_cliente) AS id_min,
    MAX(c.id_cliente) AS id_max,
    COUNT(DISTINCT p.id_cliente) AS compradores_com_cadastro
  FROM workspace.default.ecommerce_clientes c
  LEFT JOIN (SELECT DISTINCT id_cliente FROM workspace.default.silver_ecommerce_pedidos) p
    ON c.id_cliente = p.id_cliente

  UNION ALL

  SELECT
    'sqlserver_catalog.dbo.ecommerce_clientes' AS tabela_fonte,
    COUNT(*) AS total_clientes,
    MIN(c.id_cliente) AS id_min,
    MAX(c.id_cliente) AS id_max,
    COUNT(DISTINCT p.id_cliente) AS compradores_com_cadastro
  FROM sqlserver_catalog.dbo.ecommerce_clientes c
  LEFT JOIN (SELECT DISTINCT id_cliente FROM workspace.default.silver_ecommerce_pedidos) p
    ON c.id_cliente = p.id_cliente
) fontes
ORDER BY total_clientes DESC;

SELECT * FROM workspace.default.vw_fontes_compradores;